In [4]:
"""
=============================================================================
COMPONENT II: Transformer Based Sequential Data Generation
=============================================================================
"""

import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# STEP 1: Load Dataset (same as Component I)
# ─────────────────────────────────────────────
text_data = """
artificial intelligence systems learn patterns from data.
sequence models process information step by step.
recurrent neural networks are useful for sequence prediction.
lstm networks handle long term dependencies.
deep learning models improve sequence learning.
generative models create new samples from learned patterns.
language models predict the next word in a sentence.
sequence generation is used in chatbots and assistants.
machine learning helps computers learn automatically.
training data improves model accuracy.
neural networks simulate human brain structures.
optimization algorithms improve learning efficiency.
technology is transforming modern education.
online learning platforms use artificial intelligence.
students benefit from intelligent tutoring systems.
automation improves productivity and decision making.
""".strip()

print("=" * 60)
print("COMPONENT II: Transformer Based Sequential Data Generation")
print("=" * 60)

# ─────────────────────────────────────────────
# STEP 2: Word-level Tokenization
# ─────────────────────────────────────────────
tokens       = text_data.split()
vocab        = sorted(set(tokens))
word2idx_t   = {w: i + 1 for i, w in enumerate(vocab)}   # 0 = <PAD>
word2idx_t["<PAD>"] = 0
idx2word_t   = {i: w for w, i in word2idx_t.items()}
vocab_size_t = len(word2idx_t)
encoded_words = [word2idx_t[w] for w in tokens]

print(f"\n[Dataset] Total tokens     : {len(tokens)}")
print(f"[Dataset] Vocabulary size  : {vocab_size_t} unique words  (+1 for <PAD>)")
print(f"[Dataset] Sample mapping   : "
      f"'learning' -> {word2idx_t.get('learning', '?')}, "
      f"'neural' -> {word2idx_t.get('neural', '?')}")

# ─────────────────────────────────────────────
# STEP 2b: Word-level Dataset
# ─────────────────────────────────────────────
SEQ_LEN_T  = 10
BATCH_SIZE = 32

class WordDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data    = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx: idx + self.seq_len],         dtype=torch.long)
        y = torch.tensor(self.data[idx + 1: idx + self.seq_len + 1], dtype=torch.long)
        return x, y

t_dataset = WordDataset(encoded_words, SEQ_LEN_T)
t_loader  = DataLoader(t_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"\n[Sequences] Sequence length : {SEQ_LEN_T} words")
print(f"[Sequences] Total samples   : {len(t_dataset)}")
print(f"[Sequences] Batches/epoch   : {len(t_loader)}")

# ─────────────────────────────────────────────
# STEP 3: Positional Encoding
# ─────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    """
    Injects position information using sine and cosine functions.
        PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
        PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)   # even indices
        pe[:, 1::2] = torch.cos(pos * div)   # odd  indices
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

# ─────────────────────────────────────────────
# STEP 4: Transformer Encoder Architecture
# ─────────────────────────────────────────────
class TransformerGenerator(nn.Module):
    """
    Autoregressive Transformer for word-level sequence generation.
    Uses a causal mask so each position only attends to past tokens.

    Architecture:
        Word Embedding (vocab_size x d_model)
        -> Positional Encoding
        -> Transformer Encoder (num_layers x [Self-Attn + FFN + LayerNorm])
        -> Linear projection (d_model -> vocab_size)
    """
    def __init__(self, vocab_size, d_model=128, nhead=4,
                 num_layers=3, dim_ff=256, dropout=0.1, max_len=64):
        super().__init__()
        self.d_model   = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc   = PositionalEncoding(d_model, max_len, dropout)

        enc_layer    = nn.TransformerEncoderLayer(
            d_model, nhead, dim_ff,
            dropout, batch_first=True,
            norm_first=True             # Pre-LN: more stable training
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers)
        self.fc_out  = nn.Linear(d_model, vocab_size)

    def forward(self, x, src_key_padding_mask=None):
        sz   = x.size(1)
        # Causal mask: upper-triangular of -inf prevents attending to future tokens
        mask = nn.Transformer.generate_square_subsequent_mask(
            sz, device=x.device
        )
        emb = self.embedding(x) * math.sqrt(self.d_model)
        emb = self.pos_enc(emb)
        out = self.encoder(emb, mask=mask,
                            src_key_padding_mask=src_key_padding_mask)
        return self.fc_out(out)

device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trans_model = TransformerGenerator(vocab_size_t).to(device)
total_params = sum(p.numel() for p in trans_model.parameters())

print(f"\n[Model] Architecture:")
print(f"        Embedding({vocab_size_t}, 128)")
print(f"        + PositionalEncoding(max_len=64)")
print(f"        + TransformerEncoder(layers=3, heads=4, d_ff=256)")
print(f"        + Linear(128, {vocab_size_t})")
print(f"[Model] Total params : {total_params:,}")
print(f"[Model] Device       : {device}")

# ─────────────────────────────────────────────
# STEP 5: Train the Model
# ─────────────────────────────────────────────
EPOCHS_T = 150
LR_T     = 0.001

criterion = nn.CrossEntropyLoss(ignore_index=0)   # ignore <PAD>
opt_t     = optim.Adam(trans_model.parameters(), lr=LR_T)
sched_t   = optim.lr_scheduler.CosineAnnealingLR(opt_t, T_max=EPOCHS_T)

print(f"\n[Training] Epochs   : {EPOCHS_T}")
print(f"[Training] LR       : {LR_T}  (CosineAnnealing)")
print(f"[Training] Loss fn  : CrossEntropyLoss (PAD ignored)")
print(f"\n{'Epoch':>8}  {'Avg Loss':>10}  {'LR':>12}")
print("-" * 38)

for epoch in range(1, EPOCHS_T + 1):
    trans_model.train()
    total_loss = 0.0

    for xb, yb in t_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt_t.zero_grad()
        logits = trans_model(xb)
        loss   = criterion(logits.reshape(-1, vocab_size_t), yb.reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(trans_model.parameters(), max_norm=5)
        opt_t.step()
        total_loss += loss.item()

    sched_t.step()

    if epoch % 30 == 0:
        avg_loss = total_loss / len(t_loader)
        cur_lr   = sched_t.get_last_lr()[0]
        print(f"{epoch:>8}  {avg_loss:>10.4f}  {cur_lr:>12.6f}")

# ─────────────────────────────────────────────
# Generate New Sequences (Transformer)
# ─────────────────────────────────────────────
def generate_transformer(model, seed_words, length=15, temperature=0.9):
    """
    Autoregressively generate words from a seed word list.
    At each step the full context window (last SEQ_LEN_T tokens) is fed.
    """
    model.eval()
    with torch.no_grad():
        indices = [word2idx_t.get(w, 0) for w in seed_words]

        for _ in range(length):
            window = indices[-SEQ_LEN_T:]
            inp    = torch.tensor([window], dtype=torch.long).to(device)
            logits = model(inp)
            probs  = torch.softmax(
                logits[0, -1] / temperature, dim=-1
            ).cpu().numpy()
            nxt = np.random.choice(len(probs), p=probs)
            indices.append(nxt)

    return " ".join(idx2word_t.get(i, "<UNK>") for i in indices)


print("\n" + "=" * 60)
print("EXPECTED OUTPUT — GENERATED SEQUENCES (TRANSFORMER)")
print("=" * 60)

t_seeds = [
    ["deep", "learning"],
    ["neural", "networks"],
    ["sequence", "models"],
    ["language", "models"],
]
temperatures = [0.7, 0.9]

for seed in t_seeds:
    print(f"\n{'─'*55}")
    print(f"  Seed : {seed}")
    for temp in temperatures:
        out = generate_transformer(trans_model, seed, length=12, temperature=temp)
        print(f"\n  [temperature={temp}]")
        print(f"  {out}")

COMPONENT II: Transformer Based Sequential Data Generation

[Dataset] Total tokens     : 104
[Dataset] Vocabulary size  : 84 unique words  (+1 for <PAD>)
[Dataset] Sample mapping   : 'learning' -> 41, 'neural' -> 51

[Sequences] Sequence length : 10 words
[Sequences] Total samples   : 94
[Sequences] Batches/epoch   : 3

[Model] Architecture:
        Embedding(84, 128)
        + PositionalEncoding(max_len=64)
        + TransformerEncoder(layers=3, heads=4, d_ff=256)
        + Linear(128, 84)
[Model] Total params : 419,028
[Model] Device       : cuda

[Training] Epochs   : 150
[Training] LR       : 0.001  (CosineAnnealing)
[Training] Loss fn  : CrossEntropyLoss (PAD ignored)

   Epoch    Avg Loss            LR
--------------------------------------
      30      0.0769      0.000905
      60      0.0581      0.000655
      90      0.0715      0.000345
     120      0.0599      0.000095
     150      0.0628      0.000000

EXPECTED OUTPUT — GENERATED SEQUENCES (TRANSFORMER)

──────────────